In [39]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from nltk.sentiment.vader import SentimentIntensityAnalyzer
from scipy.stats import pearsonr

In [40]:
#load the data set
new = pd.read_csv("C:/Data/new.csv")
Amazone = pd.read_csv("C:/Data/yfinance_data/Data/AMZN.csv")

In [41]:
new['date'] = pd.to_datetime(new['date'], errors='coerce')

In [42]:
new['news_date'] = new['date'].dt.date

In [43]:
trading_days = Amazone['Date'].sort_values().unique()

In [44]:
new['date'] = pd.to_datetime(new['date']).dt.tz_localize(None)
Amazone['Date'] = pd.to_datetime(Amazone['Date']).dt.tz_localize(None)

In [45]:
new['date'].isna().sum()

np.int64(1351341)

In [46]:
new = new.sort_values('date')
Amazone = Amazone.sort_values('Date')

In [48]:
Amazone['Date'] = pd.to_datetime(Amazone['Date'],errors='coerce')

In [49]:
Amazone['Date_only'] = Amazone['Date'].dt.date

In [50]:
Amazone['Date'] = pd.to_datetime(Amazone['Date']).dt.tz_localize(None)

In [51]:
Amazone['Date'].isna().sum()

np.int64(0)

In [52]:
Amazone[Amazone['Date'].isna()]

,Date,Close,High,Low,Open,Volume,Date_only


In [53]:
new['date'].isna().sum()

np.int64(1351341)

In [54]:
new = new.dropna(subset=['date'])
Amazone = Amazone.dropna(subset=['Date'])

In [55]:
new['date'] = pd.to_datetime(new['date'])
Amazone['Date'] = pd.to_datetime(Amazone['Date'])

In [57]:
new = new.sort_values('date')
Amazone = Amazone.sort_values('Date')

In [58]:
result = pd.merge_asof(
    new,
    Amazone,
    left_on='date',
    right_on='Date'
)

In [ ]:
trading_days = pd.to_datetime(Amazone['Date']).drop_duplicates().sort_values()

s = pd.Series(trading_days, index=trading_days)

def align(date):
    date = pd.to_datetime(date)
    future_days = trading_days[trading_days >= date]
    return future_days.iloc[0] if len(future_days) else None

new['aligned_date'] = new['date'].apply(align)

In [ ]:
#Apply sentiment scoring
from nltk.sentiment.vader import SentimentIntensityAnalyzer

sia = SentimentIntensityAnalyzer()

new['sentiment'] = new['headline'].apply(
    lambda x: sia.polarity_scores(x)['compound']
)

In [ ]:
#Aggregate Daily Sentiment
daily_sentiment = new.groupby(['stock', 'aligned_date'])['sentiment'].mean().reset_index()

In [ ]:
#Calculate Daily Stock Returns
Amazone = Amazone.sort_values('Date')

Amazone['daily_return'] = Amazone['Close'].pct_change() * 100

In [ ]:
#Merge Sentiment + Stock Data
merged = pd.merge(
    daily_sentiment,
    Amazone,
    left_on='aligned_date',
    right_on='Date',
    how='inner'
)

In [ ]:
#Correlation Analysis
from scipy.stats import pearsonr

corr, p_value = pearsonr(merged['sentiment'], merged['daily_return'])

print("Correlation:", corr)
print("P-value:", p_value)

In [ ]:
#Scatter Plot
import matplotlib.pyplot as plt

plt.scatter(merged['sentiment'], merged['daily_return'])
plt.xlabel("Sentiment Score")
plt.ylabel("Daily Return (%)")
plt.title(f"Sentiment vs Stock Return (corr={corr:.2f})")
plt.axhline(0, linestyle="--")
plt.axvline(0, linestyle="--")
plt.show()

In [ ]:
#Sentiment Classification
def label_sentiment(x):
    if x > 0.05:
        return "Positive"
    elif x < -0.05:
        return "Negative"
    return "Neutral"

merged['sentiment_label'] = merged['sentiment'].apply(label_sentiment)

In [ ]:
#Bar Chart (Average Return per Sentiment)
avg_returns = merged.groupby('sentiment_label')['daily_return'].mean().reset_index()

In [ ]:
plt.bar(avg_returns['sentiment_label'], avg_returns['daily_return'])
plt.title("Average Stock Return by Sentiment")
plt.xlabel("Sentiment Category")
plt.ylabel("Average Return (%)")
plt.show()